# Swiggy/Zomato Delivery Analytics

Author: Neethu Raj  
Tools Used: Pandas, Python, SQL, Excel

This project analyzes food delivery operations using Python (Pandas), SQL, and Excel dashboards.

The analysis focuses on:
- order trends
- delivery performance
- rider efficiency
- cancellation behavior
- distance-based operational insights

## Contents
1. Import Libraries
2. Load Dataset
3. Data Cleaning
4. Feature Engineering
5. Exploratory Data Analysis
6. Key Insights
7. Export Cleaned Dataset

# 1. Import Libraries
The required Python libraries are imported for data cleaning, analysis, and datetime operations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 2. Load Dataset
The dataset is loaded into Pandas for preprocessing and exploratory analysis.

In [ ]:
swiggy_zomato_order = pd.read_csv("Swiggy Zomato Order Information.csv")

In [ ]:
swiggy_zomato_order.head()

# 3. Data Cleaning
Data cleaning is performed to improve data quality and ensure accurate analysis. This includes 
- handling duplicates
- checking missing values
- converting columns into appropriate data types.

##  Remove Duplicates

In [ ]:
swiggy_zomato_order = swiggy_zomato_order.drop_duplicates()

## Convert Text to Datetime
Datetime columns are converted into proper datetime format to enable time-based analysis such as hourly and daily trends.


In [ ]:
date_columns = ['order_time','order_date','allot_time','accept_time','pickup_time','delivered_time','cancelled_time']

In [ ]:
swiggy_zomato_order[date_columns] = swiggy_zomato_order[date_columns].apply(
    lambda col: pd.to_datetime(col, errors='coerce')
)

## Check Missing Values

In [ ]:
swiggy_zomato_order = swiggy_zomato_order.dropna(subset=['order_time'])

# 4. Feature Engineering

New features are created from existing columns to support operational analysis and KPI tracking. These features help identify delivery trends, rider performance, and time-based patterns.
  1. delivery_minutes
  2. order_hour

##  Delivery Time Calculation
Delivery duration is calculated to measure operational efficiency and analyze rider performance.

In [ ]:
swiggy_zomato_order['delivery_minutes'] = (
    (swiggy_zomato_order['delivered_time'] - swiggy_zomato_order['allot_time'])
    .dt.total_seconds() / 60
)

##  Order Hour Extraction
Order hour is extracted from timestamps to analyze peak demand periods throughout the day.

In [ ]:
swiggy_zomato_order['order_hour'] = swiggy_zomato_order['order_time'].dt.hour

##  Distance Calculation

In [ ]:
swiggy_zomato_order["distance"] = swiggy_zomato_order["first_mile_distance"] + swiggy_zomato_order["last_mile_distance"]

##  Distance Bucket Creation

In [ ]:
swiggy_zomato_order["distance_bucket"] = "0-2 km"
swiggy_zomato_order.loc[(swiggy_zomato_order["distance"] > 2) & (swiggy_zomato_order["distance"] <= 5),"distance_bucket"] = "2-5 km"
swiggy_zomato_order.loc[(swiggy_zomato_order["distance"] > 5) & (swiggy_zomato_order["distance"] <= 8),"distance_bucket"] = "5-8 km"
swiggy_zomato_order.loc[(swiggy_zomato_order["distance"] > 8) & (swiggy_zomato_order["distance"] <= 15),"distance_bucket"] = "8-15 km"
swiggy_zomato_order.loc[swiggy_zomato_order["distance"] > 16,"distance_bucket"] = "> 15 km"

# 5. Exploratory Data Analysis
Exploratory analysis is performed to identify trends, patterns, and operational insights from the delivery dataset.

## 5.1   Rider Performance Analysis

Rider performance analysis helps identify delivery efficiency variations among riders based on delivery duration and workload..

### Rider Efficiency Table

Create a rider efficiency table containing total orders, delivered orders, cancellations, average delivery time, and cancellation rate for each rider.

In [ ]:
rider_efficiency = swiggy_zomato_order.groupby('rider_id').agg(
    total_orders=('order_id', 'count'),
    delivered_orders=('delivered_time', 'count'), 
    total_cancelled=('cancelled', 'sum'),
    avg_delivery_time=('delivery_minutes', 'mean'),
    cancellation_rate=('cancelled', lambda x: round(x.mean() * 100, 2))
)

In [ ]:
rider_efficiency = rider_efficiency.reset_index()

Round average delivery time to 2 decimal places.

In [ ]:
rider_efficiency['avg_delivery_time'] = rider_efficiency['avg_delivery_time'].round(2)

Create a new column 'rider_category' to analyze rider performance based on avg_delivery_time.

In [ ]:
def categorize_rider(order):
    if pd.isna(order['avg_delivery_time']):
        return 'No Delivery'
    if order['avg_delivery_time'] <= 20 and order['cancellation_rate'] < 5:
        return 'Top Performer'
    if order['avg_delivery_time'] <= 30:
        return 'Average Performer'
    return 'Poor Performer'

rider_efficiency['rider_category'] = rider_efficiency.apply(categorize_rider, axis=1)

### Top 10 Riders

Create table top riders

In [ ]:
def top_10_riders(df):

    top = df[
        (df['rider_category'] == 'Top Performer') &
        (df['total_orders'] >= 10)
    ].copy()

    top = top.sort_values(
        by=['avg_delivery_time', 'cancellation_rate'],
        ascending=[True, True]
    )

    top['Rank'] = range(1, len(top) + 1)

    return top.head(10)

top_riders = top_10_riders(rider_efficiency)

In [ ]:
top_riders

Certain riders consistently achieve lower delivery times, indicating higher operational efficiency.

###  Worst 10 riders 

In [ ]:
worst_riders = rider_efficiency[
    (rider_efficiency['avg_delivery_time'] > 30) &
    (rider_efficiency['total_orders'] >= 10) &
    (rider_efficiency['cancellation_rate'] > 10)
].copy()

worst_riders = worst_riders.sort_values(
    by=['avg_delivery_time', 'cancellation_rate'],
    ascending=[False, False]
)

worst_riders.head(10)

Some riders experience significantly higher delivery times, which may indicate operational challenges or route inefficiencies.

## 5.2 Late Delivery Analysis

Late delivery percentage is analyzed to measure service reliability and identify operational inefficiencies.


### Late Delivery Percentage

In [ ]:
def late_delivery_percentage(df):

    df = df.copy()

    df['is_late'] = (
        (df['delivery_minutes'] > 30) &
        (df['delivered_time'].notna())
    )

    late_deliveries = df.groupby('rider_id').agg(
        delivered_orders=('delivered_time', 'count'),
        late_orders=('is_late', 'sum')
    )

    late_matrix = late_deliveries[
        late_deliveries['delivered_orders'] > 10
    ].copy()

    late_matrix['late_delivery_percentage'] = (
        late_matrix['late_orders'] * 100 /
        late_matrix['delivered_orders']
    ).round(2)

    late_matrix['rank'] = (
        late_matrix['late_delivery_percentage']
        .rank(method='dense', ascending=False)
    )

    return late_matrix.sort_values(
        by='late_delivery_percentage',
        ascending=False
    )


late_delivery_rider = late_delivery_percentage(swiggy_zomato_order)
  

In [ ]:
late_delivery_rider

A noticeable percentage of deliveries are delayed, indicating opportunities for improving operational planning and rider allocation.

## 5.3 Rider Productivity Analysis

### Orders Per Rider Analysis

Orders per rider are analyzed to understand workload distribution and rider productivity across operational periods.

In [ ]:
order_per_rider_per_day = (
    swiggy_zomato_order
    .groupby(['rider_id','order_date'])
    .agg(total_orders = ('order_id','count'))
    .sort_values(['rider_id','order_date'])).reset_index()
order_per_rider_per_day

###  Average Daily Orders Per Rider

Average daily orders per rider are calculated to evaluate rider productivity and delivery efficiency.

In [ ]:
avg_daily_order_per_rider = (
    swiggy_zomato_order
    .groupby('rider_id')
    .agg(
        total_orders=('order_id', 'count'),
        avg_daily_orders=(
            'order_date',
            lambda x: round(len(x) / x.nunique(), 2)
        )
    )
)

avg_daily_order_per_rider = (
    avg_daily_order_per_rider[
        avg_daily_order_per_rider['total_orders'] >= 10
    ]
    .sort_values(
        by='avg_daily_orders',
        ascending=False
    )
    .drop(columns='total_orders')
    .reset_index()
)

avg_daily_order_per_rider

Average daily order distribution provides insight into rider utilization and operational capacity management.

## 5.4 Hourly Operational Analysis

Hourly operational KPIs are calculated to analyze customer demand patterns, delivery efficiency, and cancellation behavior throughout the day.

### Hourly KPI Table

In [ ]:
order_hour_table = swiggy_zomato_order.groupby('order_hour').agg(
    total_orders=('order_id', 'count'),
    delivered_orders=('delivered_time', 'count'), 
    total_cancelled=('cancelled', 'sum'),
    avg_delivery_time=('delivery_minutes', 'mean'),
    cancellation_rate=('cancelled', lambda x: round(x.mean() * 100, 2))
).reset_index()

In [ ]:
order_hour_table["avg_delivery_time"] = order_hour_table["avg_delivery_time"].round(2)

In [ ]:
order_hour_table.head()

The hourly KPI table helps identify peak operational periods, service delays, and customer demand fluctuations across different hours.

### Peak Demand Hours

Peak demand hours are identified to understand customer ordering behavior and operational workload distribution.

In [ ]:
peak_hours = order_hour_table.sort_values(
    by='total_orders',
    ascending=False
).head(5)

peak_hours

The results indicate that customer demand is concentrated during afternoon and evening periods, increasing operational workload during these hours.

### Highest Cancellation Hours

Cancellation trends are analyzed to identify operational stress periods and potential service inefficiencies.

In [ ]:
high_cancel_hours = order_hour_table.sort_values(
    by='cancellation_rate',
    ascending=False
).head(5)

high_cancel_hours

Higher cancellation rates during certain hours may indicate rider shortages, delayed deliveries, or increased operational pressure.

### Slowest Delivery Hours

Average delivery time is analyzed across different hours to identify periods with reduced operational efficiency.

In [ ]:
slow_delivery_hours = order_hour_table.sort_values(
    by='avg_delivery_time',
    ascending=False
).head(5)

slow_delivery_hours

Longer delivery durations during peak periods suggest increased rider workload and operational congestion.

## 5.5 Distance Analysis

Distance analysis is performed to evaluate how delivery distance impacts delivery duration and operational efficiency.

### Distance KPI Table

In [ ]:
order_distance_table = swiggy_zomato_order.groupby('distance_bucket').agg(
    total_orders=('order_id', 'count'),
    delivered_orders=('delivered_time', 'count'), 
    total_cancelled=('cancelled', 'sum'),
    avg_delivery_time=('delivery_minutes', 'mean'),
    cancellation_rate=('cancelled', lambda x: round(x.mean() * 100, 2))
).reset_index()

In [ ]:
order_distance_table

The distance analysis table highlights the relationship between delivery distance, operational efficiency, and cancellation behavior.

The following visualizations help compare operational KPIs across different delivery distance ranges.

In [ ]:
order_distance_table.plot(
    x = 'distance_bucket',
    y = ["total_orders","delivered_orders","total_cancelled"],
    kind = 'line',
    marker = 'o',
    figsize=(8, 5),
    title='orders_by_distance_analysis')
plt.ylabel('Units Sold')
plt.xlabel('Distance Bucket')

In [ ]:
order_distance_table.plot(
    x = 'distance_bucket',
    y = ["cancellation_rate"],
    kind = 'line',
    marker = 'o',
    figsize=(8, 5),
    title='Cancellation Rate by Distance')
plt.ylabel('Cancellation rate')
plt.xlabel('Distance Bucket')

In [ ]:
order_distance_table.plot(
    x = 'distance_bucket',
    y = ["avg_delivery_time"],
    kind = 'line',
    marker = 'o',
    figsize=(8, 5),
    title='Delivery time by distance')
plt.ylabel('Average delivery time (min)')
plt.xlabel('Distance Bucket')

Delivery time increases progressively with distance, highlighting the operational impact of long-distance deliveries.

# 6. Key Insights

- Peak order demand occurs during afternoon and evening hours.
- Longer delivery distances increase delivery time significantly.
- Certain riders consistently outperform others in efficiency.
- Rider workload varies considerably across different days.
- Delivery completion percentage indicates overall operational efficiency.

#  7. Business Recommendations
- Increase rider allocation during peak demand hours.
- Monitor long-distance deliveries to reduce delays.
- Investigate causes of higher cancellations during busy periods.
- Reward consistently efficient riders.
- A noticeable percentage of orders experience delayed delivery, indicating potential operational bottlenecks during peak periods.

# 8. Conclusion

This project analyzed food delivery operations using Pandas, SQL, and Excel dashboards to identify operational trends, rider efficiency, delivery delays, and customer demand patterns. The analysis provides business insights that can support operational optimization and service improvement.

# 9. Export Cleaned Dataset

The cleaned and transformed dataset is exported for dashboard creation and further business analysis.

In [ ]:
swiggy_zomato_order.to_csv(
    'cleaned_swiggy_zomato_orders.csv',
    index=False
)